In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev", choices=["dev", "qa", "prd"], label = "Select Environment")
env = dbutils.widgets.get("environment")
print(env)

catalog = f"saleslake_{env}"

bronzeTable = f"{catalog}.bronze_schema_{env}.rawRegion"
silverTable = f"{catalog}.silver_schema_{env}.cleanedRegion"
print(f"Source Table : {bronzeTable}")
print(f"Target Table : {silverTable}")

srcFileLoc=f"/Volumes/{catalog}/silver_{env}/vol_saleslake_src_files_{env}/daily_region/"
# print(srcFileLoc)

In [0]:
spark.sql(f"""
INSERT INTO {silverTable}
SELECT DISTINCT
    CAST(TRIM(region_id) AS INT)              AS region_id,
    UPPER(TRIM(region_code))                  AS region_code,
    UPPER(TRIM(region_name))                  AS region_name,
    UPPER(TRIM(country))                      AS country,
    UPPER(TRIM(sub_region))                   AS sub_region,
    TRIM(regional_manager)                    AS regional_manager,
    TO_DATE(TRIM(created_date), 'yyyy-MM-dd') AS created_date,
    CURRENT_TIMESTAMP() AS ingest_ts
FROM {bronzeTable}
WHERE ingest_ts > (
    SELECT COALESCE(MAX(ingest_ts), TO_TIMESTAMP("1990-01-01"))
    FROM {silverTable}
)
ORDER BY region_id
""")

print(f"Silver load complete for {silverTable}")
print("Region Silver Load Completed Successfully")

In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {silverTable}").display()

# Region Record Count
spark.sql(f"""
SELECT 
    COUNT(*) AS total_regions
FROM {silverTable}
""").show()

In [0]:
# Duplicate Region Check
# Each region_id should exist only once in Silver.

spark.sql(f"""
SELECT
    region_id,
    COUNT(*) AS duplicate_count
FROM {silverTable}
GROUP BY region_id
HAVING COUNT(*) > 1
""").show()

In [0]:
# Null Region ID Check
spark.sql(f"""
SELECT
    COUNT(*) AS null_region_id
FROM {silverTable}
WHERE region_id IS NULL
""").show()

In [0]:
# Country Distribution
# Business validation to ensure country standardization.

spark.sql(f"""
SELECT
    country,
    COUNT(*) AS region_count
FROM {silverTable}
GROUP BY country
ORDER BY region_count DESC
""").show()

In [0]:
# Sample Cleaned Data
spark.sql(f"""
SELECT *
FROM {silverTable}
ORDER BY region_id
LIMIT 10
""").show(truncate=False)